# Microsoft (MSFT) — Monte Carlo DCF Valuation

**Part 2** of the MSFT valuation project. The original DCF (see `MSFT_DCF_Report.docx` / companion repo) produced a single deterministic implied share price of **$293.08** against an actual market price of **$373.02**.

A single point estimate implicitly overstates precision — every input feeding it (WACC, terminal growth, revenue growth, CapEx trajectory) is a judgment call with real uncertainty. This notebook replaces each key deterministic assumption with a probability distribution and runs 10,000 simulations to produce a full distribution of plausible outcomes, rather than one number.

**Valuation date:** June 30, 2026 &nbsp;|&nbsp; **Iterations:** 10,000 &nbsp;|&nbsp; **Seed:** 42 (fixed, for reproducibility)

In [ ]:
import sys
sys.path.append("../src")

from model import run_dcf
from simulate import run_simulation
from analyze import summarize, print_summary, plot_distribution, variance_contribution, plot_tornado

## 1. Validate: deterministic model reproduces the original Excel output

In [ ]:
base_price = run_dcf()
print(f"Deterministic base-case implied price: ${base_price:,.2f}")
print("Should match the original Excel DCF's $293.08.")

## 2. Run the Monte Carlo simulation

In [ ]:
df = run_simulation(n=10000, seed=42)
df.to_csv("../outputs/simulation_results.csv", index=False)
df.head()

## 3. Summary statistics

In [ ]:
summary = summarize(df)
print_summary(summary)

## 4. Distribution of implied share price

In [ ]:
plot_distribution(df)

## 5. Which assumptions actually drive the valuation?

Spearman rank correlation between each sampled input and the resulting implied price — a more rigorous view of sensitivity than a two-way Excel data table, since it accounts for every assumption moving simultaneously rather than flexing two at a time.

In [ ]:
corr = variance_contribution(df)
print(corr)
plot_tornado(corr)

## Key takeaway

WACC and near-term CapEx intensity dominate the variance in the output — far more than revenue growth assumptions. This is consistent with the underlying economics: with the terminal value representing the large majority of enterprise value, the discount rate applied to it matters more than almost any single operating assumption. It also validates that the deepest analytical effort in the original deterministic build (WACC construction, CapEx guidance vs. trend) was correctly placed on the inputs that matter most.